#### Algorithm

Two complementary methods are implemented side-by-side:

| Method | Formula | Strengths | Weaknesses |
|---|---|---|---|
| **Wilson Score** (primary) | Lower bound of 95 % CI for $\hat{p} = (\text{rating}-1)/4$ | Statistically robust; penalises sparse ratings correctly; used by Reddit & IMDB | Slightly complex; favours established places at high confidence |
| **Bayesian Average** (reference) | $\frac{v}{v+m} \cdot R + \frac{m}{v+m} \cdot C$ | Simple; naturally regresses to global mean | Less principled; sensitive to choice of $m$ |

**Wilson Score intuition:** a place with a 5.0 average from 3 reviews is *less trustworthy* than one with 4.8 from 400 reviews. The lower bound of the confidence interval captures exactly that — the more reviews, the tighter the interval and the higher the lower bound.

The `confidence` parameter controls how conservative the ranking is:
- **0.99** (default) — favours well-established places with many reviews  
- **0.95** — gives newer high-rated places a larger boost


In [9]:
import numpy as np
import pandas as pd
# ── Wilson Score lower bound ─────────────────────────────────────────────────
# Treats mean star rating (1–5) as a proportion p = (rating - 1) / 4 ∈ [0, 1],
# then returns the lower bound of the Wilson confidence interval.
# A higher confidence makes the ranking more conservative (favours many reviews).
# Reference: https://mattsayar.com/where-are-the-best-restaurants-in-my-city-a-statistical-analysis/

Z_TABLE = {0.90: 1.645, 0.95: 1.960, 0.99: 2.576}
def wilson_score(rating: float, count: int, confidence: float = 0.99) -> float:
    if count == 0:
        return 0.0
    z = Z_TABLE.get(confidence, 2.576)
    p = (rating - 1) / 4.0          # normalise [1, 5] → [0, 1]
    n = count
    return (
        p + z**2 / (2 * n) - z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))
    ) / (1 + z**2 / n)

# ── Bayesian Average (IMDb-style) ─────────────────────────────────────────────
# WR = v/(v+m) * R  +  m/(v+m) * C
# v = item's review count, R = item's mean rating
# m = prior count (median review count across dataset)
# C = global mean rating across all items

def add_bayesian_avg(df: "pd.DataFrame", rating_col="rating", count_col="userRatingCount") -> "pd.Series":
    C = df[rating_col].mean()                   # global mean
    m = df[count_col].median()                  # prior count
    v = df[count_col]
    R = df[rating_col]
    return (v / (v + m)) * R + (m / (v + m)) * C


In [10]:
CONFIDENCE = 0.95   # ← adjust: 0.99 = conservative, 0.95 = give newcomers more credit
df_ready = pd.read_csv('./ready.csv')

df_ranked = df_ready.dropna(subset=["rating", "userRatingCount"]).query("userRatingCount > 0").copy()
# display(df_ranked)

df_ranked["wilson_score"] = df_ranked.apply(
    lambda r: wilson_score(r["rating"], int(r["userRatingCount"]), CONFIDENCE), axis=1
)
df_ranked["wilson_quantile"] = df_ranked["wilson_score"].rank(pct=True)
# df_ranked["bayesian_avg"] = add_bayesian_avg(df_ranked)

df_ranked.sort_values("wilson_score", ascending=False, inplace=True)
df_ranked.reset_index(drop=True, inplace=True)
df_ranked.index += 1  # 1-based rank index

print(f"Total ranked: {len(df_ranked)}  |  confidence: {CONFIDENCE}")
df_ranked.to_csv('./ranked.csv', index=False)
display(df_ranked[(df_ranked["wilson_quantile"] >= 0.9) & (df_ranked["takeaway"] == "Dine-In")][["displayName", "primaryTypeDisplayName", "wilson_score", "wilson_quantile"]])

Total ranked: 175  |  confidence: 0.95


,displayName,primaryTypeDisplayName,wilson_score,wilson_quantile
1,Popeyes Louisiana Chicken,Chicken Restaurant,0.968333,1.000000
2,Slab City Pizza,Pizza Restaurant,0.954695,0.994286
3,Mate’s Clapham Restaurant,Restaurant,0.952401,0.988571
4,Brorritos - Clapham,Mexican Restaurant,0.951307,0.982857
5,Brekkie Cafe & Restaurant,Restaurant,0.943563,0.977143
6,Berry & Maple,Brunch Restaurant,0.942711,0.971429
7,Trinity,Fine Dining Restaurant,0.934978,0.965714
8,Slurp & Savour 鄂了嗦粉,Chinese Noodle Restaurant,0.934429,0.960000
9,Red Vine Restaurant & Cocktail Lounge & Late B...,Restaurant,0.932209,0.954286
10,Cher Thai Eatery,Thai Restaurant,0.931085,0.948571


In [ ]:
import numpy as np

# ── Competition-Adjusted Wilson Score ─────────────────────────────────────────
#
# Boosts restaurants that achieved their Wilson Score in a region where their
# cuisine is over-represented (i.e. they beat more same-type competitors).
#
# adjusted_score = wilson_score × (1 + α × ln(max(ρ, 1)))
#   ρ = P(type | region) / P(type | global)   — representation ratio
#   α — controls strength of competition boost (0 = pure Wilson, 1 = full boost)
#
# Works with a single region today (ρ = 1 everywhere → no change).
# Add a "region" column to df_ranked to unlock multi-region adjustment.

ALPHA = 0.3   # ← tune: 0.1 = subtle, 0.5 = aggressive

TYPE_COL   = "summaryType"
REGION_COL = "region"      # set to None if column doesn't exist yet

def add_competition_adjusted_score(df: pd.DataFrame, type_col=TYPE_COL, region_col=REGION_COL, alpha=ALPHA) -> pd.DataFrame:
    df = df.copy()

    # If no region column, treat everything as one region (ρ = 1, no adjustment)
    if region_col not in df.columns:
        df["_region"] = "all"
        region_col = "_region"

    total = len(df)

    # Global type proportions P(T | global)
    global_type_freq = df[type_col].value_counts(normalize=True).rename("p_global")

    # Local (region-level) type proportions P(T | R)
    region_type_counts = df.groupby([region_col, type_col]).size().rename("n_local")
    region_totals      = df.groupby(region_col).size().rename("n_region")
    local_freq = (region_type_counts / region_totals).rename("p_local").reset_index()

    # Merge onto df to get p_local and p_global for each row
    df = df.merge(local_freq, on=[region_col, type_col], how="left")
    df = df.merge(global_type_freq.reset_index().rename(columns={"index": type_col, type_col: "p_global"}),
                  on=type_col, how="left")

    # Representation ratio ρ; clamp to [ε, ∞) to avoid divide-by-zero
    eps = 1e-9
    df["representation_ratio"] = (df["p_local"] / (df["p_global"] + eps)).clip(lower=eps)

    # Boost factor: only amplify over-represented types (ρ > 1), neutral otherwise
    df["competition_boost"] = (1 + alpha * np.log(df["representation_ratio"].clip(lower=1)))

    df["adjusted_score"] = df["wilson_score"] * df["competition_boost"]

    # Clean up helper cols
    df.drop(columns=["_region"], errors="ignore", inplace=True)

    return df


df_ranked = add_competition_adjusted_score(df_ranked)

# Re-rank by adjusted_score and add a global adjusted percentile
df_ranked.sort_values("adjusted_score", ascending=False, inplace=True)
df_ranked.reset_index(drop=True, inplace=True)
df_ranked.index += 1

df_ranked["adjusted_quantile"] = df_ranked["adjusted_score"].rank(pct=True)

print(f"α = {ALPHA}  |  regions = {df_ranked.get('region', pd.Series(['all'])).nunique()}")
print(f"\nTop movers (adjusted vs wilson rank, top 30):")
top = df_ranked.head(30)[["displayName", "summaryType", "representation_ratio",
                           "wilson_score", "adjusted_score", "adjusted_quantile",
                           "shortFormattedAddress"]]
display(top)
